In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import shutil, subprocess

from lammps import lammps
EW
from ase import Atoms
from ase.io import read, write

# And lets create a starting directory
current_dir = Path.cwd() 
base = current_dir.parent
working_dir = base / "input_TEST"
working_dir.mkdir(parents=True, exist_ok=True)

potential_path = base / "potentials/mendelev03.fs"

In [2]:
# --- 2. Configuration ---
LAT_PARAM = 2.8665
"""X_SIZE = 120
Y_SIZE = 74 
Z_SIZE = 43 """

X_SIZE = 60
Y_SIZE = 32 
Z_SIZE = 20 

# Define file paths
unitcell_path = working_dir / "unitcell.lmp"
topslab_path = working_dir / "topslab.lmp"
botslab_path = working_dir / "botslab.lmp"
final_output = working_dir / "Fe_E111_110.lmp"

# Geometry math
top_deform = -0.5 / (X_SIZE + 1)
bot_deform = 0.5 / X_SIZE

# --- 3. Execute Atomsk Commands ---
print(f"Generating structure in: {working_dir}")

# Clear the working directory of all files before starting
for doomed_file in working_dir.glob("*"):
    if doomed_file.is_file():
        doomed_file.unlink()

# Create oriented unitcell
subprocess.run(["atomsk", "--create", "bcc", str(LAT_PARAM), "Fe", 
                "orient", "[111]", "[1-10]", "[11-2]", str(unitcell_path)], check=True)

# Create Top Slab
subprocess.run(["atomsk", str(unitcell_path), "-duplicate", str(X_SIZE + 1), str(int(Y_SIZE/2)), str(Z_SIZE),
                "-deform", "x", f"{top_deform:.6f}", "0.0", str(topslab_path)], check=True)

# Create Bottom Slab
subprocess.run(["atomsk", str(unitcell_path), "-duplicate", str(X_SIZE), str(int(Y_SIZE/2)), str(Z_SIZE),
                "-deform", "x", f"{bot_deform:.6f}", "0.0", str(botslab_path)], check=True)

# Merge the slabs along the Y-axis
subprocess.run(["atomsk", "--merge", "stack", "y", "2", str(botslab_path), str(topslab_path), 
                str(final_output)], check=True)

# Cleanup intermediate files
for tmp_file in [unitcell_path, topslab_path, botslab_path]:
    tmp_file.unlink(missing_ok=True)

print(f"Base structure generation complete: {final_output}")

Generating structure in: /home/Ethan/Projects/prec_interactions/input_TEST
 ___________________________________________________
|              ___________                          |
|     o---o    A T O M S K                          |
|    o---o|    Version master-2025-10-28 (Beta)     |
|    |   |o    (C) 2010 Pierre Hirel                |
|    o---o     https://atomsk.univ-lille.fr         |
|___________________________________________________|
    ( π = 3.1415926535897931 )
>>> Creating system:
..> Bcc Fe oriented X=[111], Y=[1-10], Z=[11-2].
..> System was successfully created.
>>> Writing output file(s) (6 atoms):
..> Successfully wrote LAMMPS data file: /home/Ethan/Projects/prec_interactions/input_TEST/unitcell.lmp (806B)
\o/ Program terminated successfully!
    Total time: 0.019 s.; CPU time: 0.580 s.
 ___________________________________________________
|              ___________                          |
|     o---o    A T O M S K                          |
|    o---o|    Ver

In [3]:
# Then we change the types of the atoms on the top and bottom layers (so we can easilly fix them later)
num_fixed_layers = 6 # This is the nuumber of layers we are going to fix on the top and the bottom
minimized_output = final_output.with_name(f"{final_output.stem}_slabbed.lmp")

atoms = read(final_output, format="lammps-data")

# -------------------------
# 1. Periodicity and Vacuum
# -------------------------
atoms.set_pbc([True, False, True])
atoms.center()

# -------------------------
# 2. Y-coordinates
# -------------------------
y = atoms.positions[:, 1]
y_min, y_max = y.min(), y.max()

# -------------------------
# 3. Determine layer index
# -------------------------
layer_thickness = (y_max - y_min) / Y_SIZE

# Compute integer layer index for each atom
layer_index = ((y - y_min) / layer_thickness).astype(int)

# Safety clamp (avoid rare floating-point overflow on top layer)
layer_index = np.clip(layer_index, 0, Y_SIZE - 1)

# -------------------------
# 4. Assign LAMMPS types
# -------------------------
types = np.ones(len(atoms), dtype=int)

fixed_mask = (
    (layer_index < num_fixed_layers) |
    (layer_index >= Y_SIZE - num_fixed_layers)
)

types[fixed_mask] = 2

atoms.set_array("type", types)

# -------------------------
# 5. Write LAMMPS data file
# -------------------------
write(
    working_dir / minimized_output,
    atoms,
    format="lammps-data",
    atom_style="atomic"
)

In [4]:
# Then we turn the top and bottom slabs into atoms of different types
lmp = lammps()

lmp.cmd.units("metal")
lmp.cmd.dimension(3)
lmp.cmd.boundary("p", "s", "p")
lmp.cmd.atom_style("atomic")
lmp.cmd.atom_modify("map", "yes")

# ===========================
# Interatomic potential
# ===========================
lmp.cmd.read_data(working_dir / minimized_output)
lmp.cmd.pair_style("eam/fs")
lmp.cmd.pair_coeff("* *", potential_path, "Fe", "Fe")

lmp.cmd.neighbor(2.0, "bin")
lmp.cmd.neigh_modify("delay", 10, "check", "yes")

# ===========================
# Group atom types
# ===========================
lmp.cmd.group("mobgrp", "type", 1)
lmp.cmd.group("fixgrp", "type", 2)
lmp.cmd.fix(1, "fixgrp", "setforce", "NULL", 0.0, "NULL")

# ===========================
# Energy minimisation
# ===========================
lmp.cmd.thermo_style("custom", "step", "temp", "pe", "etotal",
                        "pxx", "pyy", "pzz", "pxy", "pyz", "pxz",
                        "lx", "ly", "lz")
lmp.cmd.thermo(10)
lmp.cmd.min_style("cg")
lmp.cmd.minimize(10e-4, 10e-6, 5000, 10000)

minimized_output = final_output.with_name(f"{final_output.stem}_min.lmp")

lmp.cmd.write_data(minimized_output, "nocoeff")

LAMMPS (22 Jul 2025)
OMP_NUM_THREADS environment is not set. Defaulting to 1 thread.
  using 1 OpenMP thread(s) per MPI task
Reading data file ...
  orthogonal box = (0 0 0) to (150.18889 129.72298 140.42925)
  1 by 1 by 1 MPI processor grid
  reading atoms ...
  232320 atoms
  read_data CPU = 0.291 seconds
145200 atoms in group mobgrp
87120 atoms in group fixgrp
Switching to 'neigh_modify every 1 delay 0 check yes' setting during minimization
Neighbor list info ...
  update: every = 1 steps, delay = 0 steps, check = yes
  max neighbors/atom: 2000, page size: 100000
  master list distance cutoff = 7.3
  ghost atom cutoff = 7.3
  binsize = 3.65, bins = 42 35 39
  1 neighbor lists, perpetual/occasional/extra = 1 0 0
  (1) pair eam/fs, perpetual
      attributes: half, newton on
      pair build: half/bin/atomonly/newton
      stencil: half/bin/3d
      bin: standard
Setting up cg style minimization ...
  Unit style    : metal
  Current step  : 0
Per MPI rank memory allocation (min/avg/ma

In [5]:
num_y_layers = Y_SIZE*2
num_fixed_layers = 6

prec_radius = [20, 30, 40]

atoms_pris = read(minimized_output, format="lammps-data")

for radius in prec_radius:

    output_file = f"Fe_E111_110_R{radius}.lmp"

    if output_file == "Fe_E111_110_R0.lmp":
        output_file = "Fe_E111_110_prepped.lmp"

    atoms = atoms_pris.copy()

    # -------------------------
    # 1. Periodicity and Vacuum
    # -------------------------
    atoms.set_pbc([True, False, True])
    atoms.center(vacuum=5.0, axis=1)

    # -------------------------
    # 2. Shift atoms by half box length in X
    # -------------------------
    Lx = atoms.cell[0, 0]
    atoms.positions[:, 0] -= Lx / 2.0

    # -------------------------
    # 3. Y-layer logic for top/bottom slabs
    # -------------------------
    y = atoms.positions[:, 1]
    y_min, y_max = y.min(), y.max()
    layer_thickness = (y_max - y_min) / num_y_layers
    layer_index = ((y - y_min) / layer_thickness).astype(int)
    layer_index = np.clip(layer_index, 0, num_y_layers - 1)

    # -------------------------
    # 4. Initialize types
    # -------------------------
    types = np.ones(len(atoms), dtype=int)  # default = 1 (Middle/Mobile region)

    # Bottom slabs -> type 3 (lower layer indices)
    bottom_mask = (layer_index < num_fixed_layers)
    types[bottom_mask] = 3

    # Top slabs -> type 2 (higher layer indices)
    top_mask = (layer_index >= num_y_layers - num_fixed_layers)
    types[top_mask] = 2

    # -------------------------
    # 5. Precipitate spherical region -> type 4 (handles PBC)
    # -------------------------
    # Fractional coordinates
    frac = atoms.get_scaled_positions()  # [0,1)
    center_frac = np.array([0.5, 0.5, 0.5])  # box center

    # Minimum image convention for periodic boundaries
    delta = frac - center_frac
    delta -= np.round(delta)  # wrap into [-0.5, 0.5]

    # Convert to Cartesian distances
    cart_delta = delta * atoms.get_cell().diagonal()
    distances = np.linalg.norm(cart_delta, axis=1)

    # Assign precipitate type 4
    # Note: This will overwrite slab types if the sphere overlaps them.
    sphere_mask = distances <= radius
    types[sphere_mask] = 4

    # -------------------------
    # 6. Attach types and wrap positions
    # -------------------------
    atoms.set_array("type", types)
    atoms.wrap()

    # -------------------------
    # 7. Write LAMMPS data file
    # -------------------------
    unique_types = np.unique(types)
    write(
        working_dir / output_file,
        atoms,
        format="lammps-data",
        atom_style="atomic",
        specorder=[str(t) for t in unique_types]  # ensures header matches actual types
    )
    print(f"Ran for radius {radius}")

Ran for radius 20
Ran for radius 30
Ran for radius 40
